# Is the laminar-dominance trimodality real or an artifact?

Every subject/probe/structure shows a striking trimodality in two per-OFF quantities:

- `supra_concentration = supra_area / (supra_area + infra_area)`: sharp mass at 0 and 1
  plus a central ~0.5 peak.
- `center_of_mass_depth` (COM): a milder version of the same.

Both are lossy projections of one object, each OFF's per-pixel depth marginal. This
notebook builds a shape-preserving, depth-randomized null that isolates the single
variable of interest, each OFF's true depth position, while holding everything else
(footprint size, shape, per-channel time-occupancy, the band geometry, the channel grid)
exactly as in production. The null reuses the real measurement code verbatim
(`add_laminar_areas`, the flip-aware `laminar_concentrations`, and the COM centroid), so
the only thing it destroys is where each OFF sits along depth.

For each measure: how much of the empirical distribution is reproduced when OFFs of the
same shape are dropped at uniform random depth? Mass the null reproduces is mechanical,
a consequence of size and geometry; the residual demands a depth-occurrence explanation,
either real biology or a detection rate that varies with depth, which is a separate
follow-up.

Helper: `cnpix_local_sleep.morphological.laminar_null`. Requires NFS (reads full-48h
`morphological`).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import pubplots as pp
from joblib import Memory

from cnpix_local_sleep import files
from cnpix_local_sleep import sps_conf
from cnpix_local_sleep.morphological import mua
from cnpix_local_sleep.morphological import laminar_null as ln
import cnpix_local_sleep.morphological.pipeline.postprocess_offs as ppo
from cnpix_local_sleep import channel_anatomy

sns.set_context("notebook")
RNG_SEED = 0
N_REPS = 1          # one placement already matches event count + total OFF mass
N_CHANNELS_CONNECT = 5   # final spatial binary_closing element (detection default)
N_PERM_OCC = 200    # permutations for the occupancy null test
N_BAND_REPS = 50    # null reps for the concentration/COM sampling band

# Shared supports for the two measures.
CONC_BINS = np.linspace(0.0, 1.0, 41)
N_DEPTH_BINS = 40

# Where publication SVGs are written, and where the r-offp group-level R outputs
# (consumed by the forest-plot cell) live. Derived from the package so they
# follow the repo instead of one person's checkout.
EXTDATA = files.get_r_offp_extdata_dir()
R_OFFP = EXTDATA.parents[1]
FIG_DIR = R_OFFP.parent / "notebooks/morphological/figures/laminar_trimodality_null"
FIG_DIR.mkdir(parents=True, exist_ok=True)
R_OUTPUT_DIR = R_OFFP / "_output_depth_profile"
LAMINAR_NULL_PARQUET = EXTDATA / "summarized_depth_profile.parquet"

# Publication mode: when True, plot titles use anonymized subject labels.
PLOT_FOR_PUB = False
SUBJECT_ALIASES = {
    "CNPIX10-Charles": "Subject1",
    "CNPIX11-Adrian": "Subject2",
    "CNPIX12-Santiago": "Subject3",
    "CNPIX14-Francis": "Subject4",
    "CNPIX15-Claude": "Subject5",
    "CNPIX16-Walter": "Subject6",
    "CNPIX17-Hans": "Subject7",
    "CNPIX18-Pier": "Subject8",
    "CNPIX19-Otto": "Subject9",
    "CNPIX2-Segundo": "Subject10",
    "CNPIX4-Doppio": "Subject11",
    "CNPIX6-Eugene": "Subject12",
    "CNPIX7-Giuseppe": "Subject13",
    "CNPIX8-Allan": "Subject14",
    "CNPIX9-Luigi": "Subject15",
}


def disp_subject(subject):
    # Subject label for plots: anonymized alias when PLOT_FOR_PUB, else real name.
    return SUBJECT_ALIASES.get(subject, subject) if PLOT_FOR_PUB else subject

# On-disk cache for the expensive per-structure analysis (NFS reads + the null
# computations). `analyze_combo` is wrapped with joblib.Memory below, so a re-run
# that only touches the plotting recomputes nothing; only a combo whose inputs or
# whose `analyze_combo` SOURCE changed is recomputed. joblib hashes the function
# body but not its callees, so after editing `cnpix_local_sleep.morphological.laminar_null` (the
# null kernels) clear the cache with `memory.clear()` or set REBUILD_CACHE=True.
CACHE_DIR = Path.home() / ".cache" / "offproj_laminar_trimodality_null"
REBUILD_CACHE = False
memory = Memory(CACHE_DIR, verbose=0)
if REBUILD_CACHE:
    memory.clear(warn=False)

Caching: the per-structure analysis (`analyze_combo`) is wrapped with `joblib.Memory`,
so its results are persisted under `~/.cache/offproj_laminar_trimodality_null`. The
first run fills the cache, a few minutes of NFS reads plus the null computations; every
later run, including re-runs that only change the plotting, reloads from disk in
seconds. A combo is recomputed only if its inputs or the `analyze_combo` source change.
joblib hashes `analyze_combo` but not its callees, so clear the cache after editing
`cnpix_local_sleep.morphological.laminar_null`: set `REBUILD_CACHE = True` above, or
call `memory.clear()`.


## The 13 cortical-laminar structures (full-48h `morphological`)

In [ ]:
SPSL = sps_conf.get_subject_probe_structure_list(
    method=mua.files.METHOD,
    exclude_thalamus=True, exclude_striatum=True,
    exclude_other=True, exclude_nonlaminar=True,
)
EXAMPLE = ("CNPIX12-Santiago", "imec1", "mPPC")
SPSL = [EXAMPLE] + [c for c in SPSL if c != EXAMPLE]   # show the example first
SPSL

## Per-combo analysis

`analyze_combo` loads one structure, runs the depth-randomized null, and returns every
array the figures and the attribution table need. It also records the granular-gap vs
closing geometry: the excluded middle band is 10% of the structure span, and where that
gap (in channels) is `<= n_channels_connect`, the final spatial closing can bridge a
supra-only and an infra-only OFF into one straddling event, fabricating the central 0.5
concentration peak at the detection stage.

It takes a per-combo integer `seed` rather than a live RNG, so it is a pure function of
its arguments. That is what lets `@memory.cache` persist each structure's result on disk
and reuse it on re-run.


In [ ]:
@memory.cache
def analyze_combo(subject, probe, structure, seed):
    rng = np.random.default_rng(seed)
    offs, lbl_ixs, y = ln.load_structure_data(subject, probe, structure)
    ref = offs.set_index("label")

    # Empirical concentrations via the flip-aware SPOT (matches production).
    sc_emp, ic_emp = ppo.laminar_concentrations(
        ref, subject=subject, probe=probe, structure=structure
    )
    com_emp = ref["center_of_mass_depth"]
    # Channel extent per OFF, straight from the footprints (the full-48h
    # offs.parquet does not persist min/max channel index).
    ext_by_label = {
        lab: int(chan.max() - chan.min() + 1)
        for lab, (_t, chan) in lbl_ixs.items()
    }
    extent = np.array([ext_by_label.get(lab, 0) for lab in ref.index])

    # Collapse footprints ONCE; reuse across both placements, both null bands,
    # and both occupancy readouts (the per-OFF collapse dominates the runtime).
    collapsed = ln.collapse_footprints(lbl_ixs)

    # The whole-structure (uniform) null places OFFs over the full ANATOMICAL
    # structure (extends past the detection window) and observes only the detected
    # channels; this window is needed by occupancy_null_test (null_measures_per_off
    # and null_measure_bands derive it internally from subject/probe/structure).
    sb = ln.structure_index_bounds(y, subject, probe, structure)

    # Two mechanical nulls (same footprints, only depth position destroyed):
    #   feasible = no-clip, size-preserving: each footprint dropped uniformly among
    #              its IN-DETECTION positions. Full-span OFFs have no freedom
    #              (delta=0), so this is the clean centroid-contraction baseline for
    #              COM (clipping cannot deflate COM extremes here).
    #   uniform  = whole-structure: each footprint dropped uniformly over the full
    #              anatomical structure, observing only the detection window, so an
    #              OFF centered beyond the detected channels is seen only partially.
    #              The clipped overhang is divided out by unit-mass renormalization
    #              (shape comparison); replaces the feasible edge taper w/ leak-in.
    nul_u = ln.null_measures_per_off(lbl_ixs, y, subject, probe, structure, rng,
                                     placement="uniform", collapsed=collapsed)
    nul_f = ln.null_measures_per_off(lbl_ixs, y, subject, probe, structure, rng,
                                     placement="feasible", collapsed=collapsed)

    # Geometry: band borders + excluded-middle gap in channels.
    borders = channel_anatomy.get_layer_borders(subject, probe, structure)
    supra = borders[borders.layer == "supra"].iloc[0]
    infra = borders[borders.layer == "infra"].iloc[0]
    pitch = np.median(np.diff(np.sort(y)))
    gap_um = float(supra["lo"] - infra["hi"])
    gap_chans = gap_um / pitch

    depth_bins = np.linspace(float(y.min()), float(y.max()), N_DEPTH_BINS + 1)
    com_arr = com_emp.to_numpy()
    com_support = (float(y.min()), float(y.max()))
    # CONCENTRATION: the Wasserstein skill-score attribution 1-W1(emp,null)/W1(emp,
    # flat) is well-behaved here (support [0,1] keeps emp far from flat).
    attr_conc = ln.mechanical_attribution(
        sc_emp.to_numpy(), nul_u["supra_concentration"].to_numpy(),
        support=(0.0, 1.0), rng=rng)
    # COM: the skill-score attribution is UNRELIABLE for the feasible null (its
    # flat denominator collapses when emp COM is itself near-flat, e.g. tall
    # probes), so report robust effect sizes instead:
    #   w1_com_feasible      = W1(emp COM, feasible-null COM) in um  (magnitude)
    #   com_spread_ratio     = std(emp)/std(feasible)  (direction; >1 = emp COM
    #                          reaches extremes MORE than contraction predicts).
    # attr_com_uniform is kept only to show the original (misleading) reading.
    mech_com_f = ln.mechanical_attribution(
        com_arr, nul_f["center_of_mass_depth"].to_numpy(),
        support=com_support, rng=rng)
    attr_com_uniform = ln.mechanical_attribution(
        com_arr, nul_u["center_of_mass_depth"].to_numpy(),
        support=com_support, rng=rng)
    w1_com_feasible = mech_com_f["w_null"]
    com_std_emp = float(np.nanstd(com_arr))
    com_std_feasible = float(np.nanstd(nul_f["center_of_mass_depth"].to_numpy()))
    com_spread_ratio = com_std_emp / com_std_feasible

    # Null sampling bands (mean ±2sd): concentration under the UNIFORM null, COM
    # under the FEASIBLE null (the relevant centroid-contraction baseline).
    bands_u = ln.null_measure_bands(lbl_ixs, y, subject, probe, structure, rng,
                                    n_reps=N_BAND_REPS, conc_bins=CONC_BINS,
                                    depth_bins=depth_bins, placement="uniform",
                                    collapsed=collapsed)
    bands_f = ln.null_measure_bands(lbl_ixs, y, subject, probe, structure, rng,
                                    n_reps=N_BAND_REPS, conc_bins=CONC_BINS,
                                    depth_bins=depth_bins, placement="feasible",
                                    collapsed=collapsed)
    # Occupancy depth marginals vs the depth null, time-weighted (total OFF-time
    # per channel) and count-weighted (events touching each channel, duration-blind,
    # more sensitive to where events occur), under both null placements:
    #   feasible = no-clip, size-preserving null (footprint dropped uniformly among
    #              its in-DETECTION positions). It reproduces the detection-edge
    #              taper (large OFFs cannot center near the edge) and is the primary
    #              occupancy effect size (matches the COM null).
    #   uniform  = whole-structure null: footprints dropped uniformly over the full
    #              anatomical structure, observing only the detection window. The
    #              clipped overhang is divided out by unit-mass renormalization (the
    #              test compares depth SHAPE only), so the f-dependent
    #              partial-visibility deficit biases neither tv nor W1; this replaces
    #              the feasible edge taper with the asymmetric leak-in shape. Read it
    #              as a whole-structure robustness check; prefer feasible.
    occ = {
        (w, pl): ln.occupancy_null_test(lbl_ixs, y, rng, n_perm=N_PERM_OCC,
                                        weighting=w, placement=pl,
                                        struct_bounds=sb, collapsed=collapsed)
        for w in ("time", "count") for pl in ("feasible", "uniform")
    }

    return dict(
        subject=subject, probe=probe, structure=structure,
        n_offs=len(offs), n_chans=int(y.size), pitch=float(pitch),
        y=y, supra=supra, infra=infra, gap_um=gap_um, gap_chans=gap_chans,
        sc_emp=sc_emp.to_numpy(), com_emp=com_arr,
        extent=extent,
        sc_null=nul_u["supra_concentration"].to_numpy(),
        com_null_u=nul_u["center_of_mass_depth"].to_numpy(),
        com_null_f=nul_f["center_of_mass_depth"].to_numpy(),
        # Feasible is the primary occupancy readout; uniform retained for the
        # edge-artifact comparison.
        occ_time=occ[("time", "feasible")], occ_count=occ[("count", "feasible")],
        occ_time_u=occ[("time", "uniform")], occ_count_u=occ[("count", "uniform")],
        depth_bins=depth_bins,
        bands_u=bands_u, bands_f=bands_f, attr_conc=attr_conc,
        attr_com_uniform=attr_com_uniform, w1_com_feasible=w1_com_feasible,
        com_spread_ratio=com_spread_ratio, com_std_emp=com_std_emp,
        com_std_feasible=com_std_feasible,
    )

### Figure: the three depth marginals vs their nulls

Three panels per structure, depth on the vertical axis throughout, so the COM and the
two occupancy marginals are read on the same depth axis. Each panel is histogram-style:
the observed distribution in black, the null mean ±2sd band in orange.

1. `center_of_mass_depth` vs the feasible (centroid-contraction) null. The no-clip,
   size-preserving null pins full-span OFFs central, so it is the honest baseline for
   the COM midpoint. The title carries the robust effect size W1 (µm, empirical vs
   feasible) and the spread ratio std(emp)/std(feasible); `>1` means the empirical COM
   reaches extreme depths more than centroid contraction predicts. The skill-score
   attribution is not used for COM: its `W1(emp,flat)` denominator collapses when the
   empirical COM is itself near-flat, as on the tall probes, giving spuriously large
   negative values.
2. Time-occupancy vs the feasible depth null. Total OFF-time per channel under H0, OFFs
   placed at random in-bounds depth. Dominated by long/tall OFFs, and heavily smeared.
3. Count-occupancy vs the feasible depth null. Number of OFF events touching each
   channel, duration-blind, and more sensitive to where events occur.

Both occupancy panels show the feasible (no-clip, size-preserving, in-detection) null as
the primary baseline, with the uniform whole-structure null mean overlaid as a thin
dashed grey line. The uniform null places same-size OFFs over the full anatomical
structure, which extends past the detection window, and observes only the detected
channels, so OFFs centered beyond the detected channels are seen only partially. That
replaces the feasible null's detection-edge taper with the asymmetric leak-in shape. Its
clipped overhang is divided out by unit-mass renormalization, since the test compares
depth shape only, so the f-dependent partial-visibility deficit biases neither TV nor
W1, and its grey curve sits at the data's level rather than below it. The panel title
reports both W1s so the gap is explicit.


In [ ]:
def plot_combo(R):
    # Three depth marginals, depth on the vertical axis throughout: the COM
    # midpoint distribution vs its feasible (centroid-contraction) null, and the
    # time- and count-weighted per-channel occupancy vs the depth-uniform null.
    #
    # Publication styling via pubplots (figma destination): fonts stay at the
    # pubplots 5/6/7pt defaults (no explicit fontsize anywhere), and the figsize
    # and every explicit linewidth are routed through pp.scale() so they scale
    # with the figma factor instead of deviating from the (scaled) rc defaults.
    with pp.destination("figma"):
        fig, ax = plt.subplots(1, 3, figsize=pp.scale(6.9, 2.3))
        title = f"{disp_subject(R['subject'])} / {R['probe']} / {R['structure']}"
        fig.suptitle(
            f"{title}   (n_offs={R['n_offs']:,}, n_chans={R['n_chans']}, "
            f"gap={R['gap_chans']:.1f} ch vs closing={N_CHANNELS_CONNECT})"
        )

        # COM (depth-on-y): empirical (black step histogram) vs the FEASIBLE
        # (centroid-contraction) null mean ±2sd band. Title: W1 effect size (um) +
        # spread ratio (>1 = emp reaches extremes more than contraction predicts).
        def _com_panel(a, emp, bins, mean, sd, ylabel, title):
            ctr = 0.5 * (bins[:-1] + bins[1:])
            a.hist(emp, bins=bins, density=True, histtype="step", lw=pp.scale(2),
                   orientation="horizontal", color="k", label="empirical")
            ys = np.repeat(bins, 2)[1:-1]          # bin edges, doubled -> staircase
            a.plot(np.repeat(mean, 2), ys, color="C1", lw=pp.scale(1.4),
                   label="null mean")
            a.fill_betweenx(ctr, mean - 2 * sd, mean + 2 * sd, step="mid",
                            color="C1", alpha=0.25, label="null ±2sd")
            a.set(xlabel="density", ylabel=ylabel, title=title)
            a.legend()

        _com_panel(ax[0], R["com_emp"], R["depth_bins"],
                   R["bands_f"]["com_mean"], R["bands_f"]["com_sd"],
                   "center_of_mass_depth (um)",
                   f"COM vs feasible null: W1={R['w1_com_feasible']:.0f}um, "
                   f"spread={R['com_spread_ratio']:.2f}")

        # Per-channel depth edges (midpoints between channels, ends extended by a
        # half-step), so a per-channel profile can be drawn as a depth histogram.
        def _chan_edges(yv):
            yv = np.asarray(yv, float)
            mids = 0.5 * (yv[:-1] + yv[1:])
            return np.concatenate([[2 * yv[0] - mids[0]], mids, [2 * yv[-1] - mids[-1]]])

        # Occupancy: observed (black) vs the FEASIBLE depth null (orange, primary),
        # drawn histogram-style (per-channel staircase) to match the COM panel. The
        # UNIFORM whole-structure null mean is overlaid as a thin grey line for
        # contrast; it places OFFs over the full anatomical structure and observes
        # only the detection window, renormalized to unit mass (shape comparison), so
        # it sits at the data's level and shows the asymmetric leak-in shape that
        # replaces the feasible edge taper.
        def _occ_panel(a, occ, occ_u, label):
            # Hide the eroded extreme channels (observed occupancy is exactly 0
            # there, a detection-morphology boundary artifact, not biology; no
            # channel is truly empty). The W1/TV in the title are still computed
            # over the full detectable range (see occupancy_null_test): trimming
            # them is cosmetic, and restricting the W1 itself to the observed span
            # would wrongly erase any genuine depth-avoidance signal.
            obs = occ["obs_p"]
            nz = np.flatnonzero(obs > 0)
            sl = slice(int(nz[0]), int(nz[-1]) + 1) if nz.size else slice(None)
            yv = occ["y"][sl]
            ys = np.repeat(_chan_edges(yv), 2)[1:-1]   # channel edges -> staircase
            a.plot(np.repeat(obs[sl], 2), ys, color="k", lw=pp.scale(1.6),
                   label="observed")
            a.plot(np.repeat(occ["null_mean"][sl], 2), ys, color="C1",
                   lw=pp.scale(1.4), label="feasible null")
            a.fill_betweenx(yv, (occ["null_mean"] - 2 * occ["null_sd"])[sl],
                            (occ["null_mean"] + 2 * occ["null_sd"])[sl], step="mid",
                            color="C1", alpha=0.25, label="null ±2sd")
            a.plot(np.repeat(occ_u["null_mean"][sl], 2), ys, color="0.5",
                   lw=pp.scale(0.9), ls="--", label="uniform null")
            a.set(xlabel="occupancy (frac)", ylabel="depth (um)",
                  title=(f"{label}-occ vs feasible null: "
                         f"W1={occ['w1_um']:.1f}um (uniform "
                         f"{occ_u['w1_um']:.1f}), TV={occ['tv']:.2f}"))
            a.xaxis.set_major_locator(MaxNLocator(nbins=4))
            a.legend()

        _occ_panel(ax[1], R["occ_time"], R["occ_time_u"], "time")
        _occ_panel(ax[2], R["occ_count"], R["occ_count_u"], "count")

        fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

### Run the example structure with full detail

In [ ]:
# Per-combo seed convention: SPSL[i] gets seed RNG_SEED + i. EXAMPLE is SPSL[0],
# so it (and the cached result reused by the all-structures loop) uses RNG_SEED.
R0 = analyze_combo(*EXAMPLE, RNG_SEED)
fig = plot_combo(R0)
plt.show()
print({k: R0[k] for k in ["n_offs", "n_chans", "gap_chans"]})
print(f"concentration attribution (uniform null)={R0['attr_conc']['attribution']:.3f} "
      f"(W_null={R0['attr_conc']['w_null']:.3f}, W_flat={R0['attr_conc']['w_flat']:.3f}, "
      f"resolvable={R0['attr_conc']['resolvable']})")
print(f"COM vs FEASIBLE null: W1={R0['w1_com_feasible']:.1f}um  "
      f"spread_ratio(emp/feasible)={R0['com_spread_ratio']:.3f}  "
      f"(std_emp={R0['com_std_emp']:.1f}um, std_feasible={R0['com_std_feasible']:.1f}um)")
print(f"  [for comparison, the UNRELIABLE uniform-null COM attribution="
      f"{R0['attr_com_uniform']['attribution']:.3f}, flat-denominator artifact]")
print(f"time-occupancy  vs FEASIBLE null: W1={R0['occ_time']['w1_um']:.1f}um  "
      f"TV={R0['occ_time']['tv']:.3f}   (uniform null W1={R0['occ_time_u']['w1_um']:.1f}um, "
      f"differs via the edge placement, not the normalization)")
print(f"count-occupancy vs FEASIBLE null: W1={R0['occ_count']['w1_um']:.1f}um  "
      f"TV={R0['occ_count']['tv']:.3f}   (uniform null W1={R0['occ_count_u']['w1_um']:.1f}um)  "
      f"n_off={R0['occ_count']['n_off']:,}")

## All 13 structures: summary table

This mirrors the per-structure parquet that `off-analysis export-depth-profile-summary`
writes to r-offp, consumed by `scripts/depth_profile_summary.R`.

- `attr_conc`: concentration Wasserstein skill-score attribution
  `1 - W1(emp,uniform-null)/W1(emp,flat)`. Near 1 means the trimodal concentration is
  what size and band geometry produce under random depth. Below 1 means real laminar
  structure the symmetric null cannot make, chiefly the supra/infra asymmetry.
- `w1_com_feasible`: robust COM effect size, the earth-mover distance (µm) between the
  empirical COM and the feasible (centroid-contraction) null. The skill-score
  attribution is not used for COM, since its flat denominator collapses on near-flat and
  tall-probe COMs; this W1 is always interpretable.
- `com_spread_ratio`, directional: `std(emp)/std(feasible)`. `>1` means the empirical
  COM reaches extreme depths more than centroid contraction predicts, a real
  depth-occurrence residual beyond the geometry.
- `occ_count_feasible_w1` / `occ_time_feasible_w1` are the primary occupancy effect
  sizes against the feasible depth null (earth-mover distance, µm); `*_feasible_tv` is
  the fraction of mass that must move. `count` is events touching each channel
  (duration-blind, more sensitive); `time` is total OFF-time per channel (smeared).
  `occ_*_uniform_w1` is the whole-structure-null W1, kept as a robustness check: the
  uniform null places OFFs over the full anatomical structure and observes only the
  detection window (partial visibility), removing the feasible null's detection-edge
  taper. It differs from feasible by how far the structure exceeds the detected
  channels. The feasible columns are the canonical effect sizes. With ~10^5 OFFs the
  permutation p (`occ_count_feasible_p`) is ~0 for any real departure, so read the
  effect size.
- `gap_chans`: excluded-middle width in channels, against `n_channels_connect=5`.


In [ ]:
# Cached per combo (seed = RNG_SEED + index); the EXAMPLE (index 0) reuses the
# result already computed above. First run fills the cache (~minutes); re-runs are
# near-instant unless a combo or analyze_combo's source changed.
results = [analyze_combo(*c, RNG_SEED + i) for i, c in enumerate(SPSL)]

def _asym(occ, R):
    return ln.occupancy_asymmetry(occ["obs_p"], occ["null_mean"], occ["y"],
                                  R["subject"], R["probe"], R["structure"])

rows = []
for R in results:
    asy_c, asy_t = _asym(R["occ_count"], R), _asym(R["occ_time"], R)
    rows.append(dict(
        subject=R["subject"], probe=R["probe"], structure=R["structure"],
        n_offs=R["n_offs"], n_chans=R["n_chans"],
        gap_chans=round(R["gap_chans"], 1),
        attr_conc=round(R["attr_conc"]["attribution"], 3),
        w1_com_feasible=round(R["w1_com_feasible"], 1),
        com_spread_ratio=round(R["com_spread_ratio"], 3),
        # PRIMARY occupancy effect sizes = the feasible null; the uniform columns
        # are a labeled robustness check (differs via the edge placement rule).
        occ_count_feasible_w1=round(R["occ_count"]["w1_um"], 1),
        occ_time_feasible_w1=round(R["occ_time"]["w1_um"], 1),
        occ_count_feasible_tv=round(R["occ_count"]["tv"], 3),
        occ_time_feasible_tv=round(R["occ_time"]["tv"], 3),
        # Signed superficial(+)/deep(-) asymmetry of the empirical excess + asym/TV.
        occ_count_feasible_asym=round(asy_c["asym"], 3),
        occ_time_feasible_asym=round(asy_t["asym"], 3),
        occ_count_feasible_asym_norm=round(asy_c["asym_norm"], 3),
        occ_time_feasible_asym_norm=round(asy_t["asym_norm"], 3),
        occ_count_uniform_w1=round(R["occ_count_u"]["w1_um"], 1),
        occ_time_uniform_w1=round(R["occ_time_u"]["w1_um"], 1),
        occ_count_feasible_p=R["occ_count"]["p_global"],
    ))
table = pd.DataFrame(rows)
table

In [ ]:
# Bootstrap CIs across structures for each scalar (descriptive; the proper
# subject-clustered inference is the r-offp scripts/depth_profile_summary.R).
def boot_ci(vals, n=5000, seed=RNG_SEED):
    vals = np.asarray(vals, float)
    g = np.random.default_rng(seed)
    means = [np.mean(g.choice(vals, vals.size, replace=True)) for _ in range(n)]
    return float(np.mean(vals)), np.percentile(means, [2.5, 97.5])

for col in ["attr_conc", "w1_com_feasible", "com_spread_ratio",
            "occ_count_feasible_w1", "occ_time_feasible_w1",
            "occ_count_feasible_tv", "occ_time_feasible_tv",
            "occ_count_feasible_asym", "occ_time_feasible_asym",
            "occ_count_feasible_asym_norm", "occ_time_feasible_asym_norm",
            "occ_count_uniform_w1", "occ_time_uniform_w1"]:
    m, (lo, hi) = boot_ci(table[col])
    print(f"{col:28s}: mean={m:.3f}  95% CI=[{lo:.3f}, {hi:.3f}]")
print("count-occupancy p_global per structure:",
      sorted(table['occ_count_feasible_p'].round(4).unique()))
print(f"com_spread_ratio > 1 in {(table['com_spread_ratio'] > 1).sum()} of {len(table)} structures")

### Full-detail figure for every structure

The three-panel `plot_combo` figure (COM vs the feasible null, and the time- and
count-occupancy marginals vs the depth null) for each subject/probe/structure in
turn.

In [ ]:
for R in results:
    plot_combo(R)
    plt.show()

## Publication figure 1: count-occupancy vs null, all structures

A small-multiples grid, 4 per row, of the count-occupancy marginal for every structure:
observed events-per-channel (black) against the feasible depth null (orange mean ±2sd).
Depth is on the vertical axis. This is the per-structure detail behind the group-level
forest plot below, and shows the shape of each structure's departure from random-depth
placement.

Each title carries the unsigned effect sizes (W1, TV) plus the signed `asym`, the
superficial(+) vs deep(-) asymmetry of the empirical excess, with channels split 50/50
at the depth midpoint and flip-corrected orientation: `> 0` means more empirical
occupancy than the null toward the cortical surface, `< 0` toward the deep half. The
eroded extreme channels, where observed occupancy is exactly 0 (a detection-morphology
boundary artifact, since no channel is truly empty), are not drawn; the W1 and TV are
still taken over the full detectable range. x-axes are capped at <=5 ticks. Saved as SVG
via `pubplots.destination("figma")`, with all fonts at the pubplots 5/6/7 pt defaults.


In [ ]:
def plot_count_occupancy_grid(results, ncols=4, save=None):
    # Count-occupancy obs-vs-feasible-null small multiples (depth on the vertical
    # axis), one panel per structure. pubplots figma styling: figsize and every
    # explicit linewidth/markersize routed through pp.scale(); no explicit fonts.
    def _chan_edges(yv):
        yv = np.asarray(yv, float)
        mids = 0.5 * (yv[:-1] + yv[1:])
        return np.concatenate([[2 * yv[0] - mids[0]], mids, [2 * yv[-1] - mids[-1]]])

    with pp.destination("figma"):
        n = len(results)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=pp.scale(1.85 * ncols, 2.0 * nrows))
        axes = np.atleast_1d(axes).ravel()
        for i, (a, R) in enumerate(zip(axes, results)):
            occ = R["occ_count"]                 # feasible null (the honest one)
            # Signed superficial(+)/deep(-) asymmetry of the empirical excess (over
            # the full detectable range, like W1/TV).
            asy = ln.occupancy_asymmetry(
                occ["obs_p"], occ["null_mean"], occ["y"],
                R["subject"], R["probe"], R["structure"],
            )
            obs = occ["obs_p"]
            nz = np.flatnonzero(obs > 0)         # hide eroded zero-occupancy edges
            sl = slice(int(nz[0]), int(nz[-1]) + 1) if nz.size else slice(None)
            yv = occ["y"][sl]
            ys = np.repeat(_chan_edges(yv), 2)[1:-1]
            a.plot(np.repeat(obs[sl], 2), ys, color="k", lw=pp.scale(1.2),
                   label="observed")
            a.plot(np.repeat(occ["null_mean"][sl], 2), ys, color="C1",
                   lw=pp.scale(1.0), label="feasible null")
            a.fill_betweenx(yv, (occ["null_mean"] - 2 * occ["null_sd"])[sl],
                            (occ["null_mean"] + 2 * occ["null_sd"])[sl], step="mid",
                            color="C1", alpha=0.25)
            a.set_title(f"{disp_subject(R['subject'])} / {R['structure']}\n"
                        f"W1={occ['w1_um']:.1f}um  TV={occ['tv']:.2f}  "
                        f"asym={asy['asym']:+.3f}")
            a.xaxis.set_major_locator(MaxNLocator(nbins=4))
            if i % ncols == 0:
                a.set_ylabel("depth (um)")
            if i >= n - ncols:
                a.set_xlabel("count-occ (frac)")
        for a in axes[n:]:
            a.axis("off")
        axes[0].legend(loc="lower right")
        fig.tight_layout()
        if save is not None:
            fig.savefig(save, format="svg", bbox_inches="tight")
            print("wrote", save)
    return fig

fig = plot_count_occupancy_grid(
    results, ncols=4, save=FIG_DIR / "count_occupancy_grid.svg"
)
plt.show()

## Publication figure 2: group-level forest plots (from the R data)

The group-level inference is computed in r-offp's `scripts/depth_profile_summary.R`
(intercept-only `metric ~ 1 + (1 | subject)`, subject as the random effect, with a
subject-cluster bootstrap). The forest plots here are drawn in Python from that R output
(`depth_profile_group_summary.csv`) so they share the publication (`pubplots`/figma)
styling, but every number comes from the R model: the group mean (diamond), the lmer
Wald CI (thick), and the subject-cluster bootstrap CI (thin). Per-structure points come
from the same `summarized_depth_profile.parquet` the R script consumed.

The feasible null is the primary occupancy readout. The primary occupancy forest is a
2x2 of the feasible-null effect sizes: W1 (µm, geometry-aware and in interpretable depth
units, but scales with probe height) and TV (unitless fraction of mass that must move,
so scale-free and more comparable across structures), each for count and time. The
uniform null goes in a separate robustness figure
(`forest_occupancy_uniform_robustness.svg`); its W1 differs from feasible's because the
two place footprints differently at the probe ends. All x-axes are capped at <=5 ticks
to keep labels legible.

Where to find everything. Per-structure scalars, including `occ_count_feasible_w1` and
`occ_count_w1`: `summarized_depth_profile.parquet` in `r-offp/inst/extdata/`, one row
per `(subject, probe, structure)`, written by `off-analysis export-depth-profile-summary`
(`cnpix_local_sleep/.../pipeline/depth_profile_export.py`). The occupancy W1 and TV
themselves are computed in
`cnpix_local_sleep/.../morphological/laminar_null.py::occupancy_null_test`. The group
model, summary and diagnostics come from `Rscript scripts/depth_profile_summary.R`:
`depth_profile_models.rds` (fitted `lmer` objects), `depth_profile_group_summary.csv`
(means and CIs), `depth_profile_model_summaries.txt` (per-metric `summary()`) and
`depth_profile_diagnostics.pdf` (residual-vs-fitted plus QQ), all under
`r-offp/_output_depth_profile/`.


In [ ]:
# Group means/CIs (from the R lmer) and per-structure points (from the parquet
# the R script consumed). The forest plot is pure presentation; no model is fit
# here. Run `Rscript scripts/depth_profile_summary.R` first to (re)generate the CSV.
group = pd.read_csv(R_OUTPUT_DIR / "depth_profile_group_summary.csv")
per_struct = pd.read_parquet(LAMINAR_NULL_PARQUET)
# conc_real_feasible is a DERIVED metric (mirrors scripts/depth_profile_summary.R):
# the concentration "real residual" = 1 - feasible-null attribution. The parquet
# stores the attribution; derive the residual here so the forest can plot it.
per_struct["conc_real_feasible"] = 1 - per_struct["attr_conc_feasible"]
print("group metrics available:", list(group["col"]))


def forest_figure(metric_cols, save=None, ncols=None):
    # One forest panel per metric: per-structure points (sorted), the group mean
    # (diamond) with the lmer Wald CI (thick) and bootstrap CI (thin), and the null
    # reference line. pubplots figma styling throughout.
    combos = (per_struct["subject"].map(disp_subject) + " / "
              + per_struct["structure"]).to_numpy()
    with pp.destination("figma"):
        ncols = ncols or len(metric_cols)
        nrows = int(np.ceil(len(metric_cols) / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=pp.scale(3.4 * ncols, 2.9 * nrows))
        axes = np.atleast_1d(axes).ravel()
        for a, col in zip(axes, metric_cols):
            grow = group[group["col"] == col]
            if grow.empty:
                a.set_title(f"{col}\n(not in R summary)")
                a.axis("off")
                continue
            grow = grow.iloc[0]
            vals = per_struct[col].to_numpy()
            order = np.argsort(vals)
            yv = np.arange(len(vals))
            a.axvline(grow["ref"], ls="--", color="grey", lw=pp.scale(1), zorder=0)
            a.scatter(vals[order], yv, color="k", s=pp.scale(12), zorder=3)
            a.set_yticks(yv)
            a.set_yticklabels(combos[order])
            gy = -1.7
            a.plot([grow["boot_lo"], grow["boot_hi"]], [gy, gy], color="C3",
                   lw=pp.scale(1.4), solid_capstyle="butt", zorder=3)
            a.plot([grow["lmer_lo"], grow["lmer_hi"]], [gy, gy], color="C3",
                   lw=pp.scale(3.5), solid_capstyle="butt", zorder=4)
            a.plot(grow["mean"], gy, marker="D", color="C3", ms=pp.scale(6),
                   zorder=5)
            a.axhline(-0.6, color="grey", lw=pp.scale(0.6))
            a.set_ylim(-2.6, len(vals) - 0.4)
            a.set_title(grow["label"])
            a.set_xlabel("effect size")
            a.xaxis.set_major_locator(MaxNLocator(nbins=4))  # <=5 x ticks
        for a in axes[len(metric_cols):]:
            a.axis("off")
        fig.tight_layout()
        if save is not None:
            fig.savefig(save, format="svg", bbox_inches="tight")
            print("wrote", save)
    return fig


# PRIMARY occupancy forest: the FEASIBLE null (the canonical effect sizes). 2x2 =
# {W1 (um, geometry-aware/interpretable), TV (unitless, scale-free -> the more
# cross-structure-comparable readout)} x {count, time}. This is the figure for the
# paper.
forest_figure(
    ["occ_count_feasible_w1", "occ_time_feasible_w1",
     "occ_count_feasible_tv", "occ_time_feasible_tv"],
    ncols=2, save=FIG_DIR / "forest_occupancy.svg",
)
plt.show()

In [ ]:
# Robustness only: the same occupancy effect under the uniform null, whose W1
# differs from feasible in a structure-dependent way (different edge placement, not
# the normalization). Not the primary readout.
forest_figure(
    ["occ_count_w1", "occ_time_w1"],
    ncols=2, save=FIG_DIR / "forest_occupancy_uniform_robustness.svg",
)
plt.show()

### Occupancy asymmetry: superficial vs deep (directional)

A signed companion to the unsigned W1 and TV: does the empirical occupancy excess over
the feasible null sit toward the superficial (cortical-surface) or the deep half of each
structure? Channels are split 50/50 at the depth midpoint, with flip-corrected
orientation so the sign means the same thing across structures.

- `asym` (top row) = (superficial-half observed mass fraction) - (superficial null
  fraction), in [-TV, TV]. `> 0` means empirical excess toward the surface, `< 0` toward
  the deep half.
- `asym / TV` (bottom row), in [-1, 1]: the share of the displaced occupancy mass that
  is one-sided. ±1 means the entire departure is on one side of the midpoint, 0 means it
  is symmetric about it.

The reference line at 0 is no net superficial/deep bias. Group mean (diamond) with the
lmer Wald and subject-cluster bootstrap CIs, as in the other forests.


In [ ]:
forest_figure(
    ["occ_count_feasible_asym", "occ_time_feasible_asym",
     "occ_count_feasible_asym_norm", "occ_time_feasible_asym_norm"],
    ncols=2, save=FIG_DIR / "forest_occupancy_asymmetry.svg",
)
plt.show()

In [ ]:
# COM + concentration forest (the other two laminar measures).
forest_figure(
    ["w1_com_feasible", "com_spread_ratio_feasible", "conc_real_feasible"],
    ncols=3, save=FIG_DIR / "forest_com_concentration.svg",
)
plt.show()

## Interpretation (draft)

The reading below is provisional and not settled. The operational facts, that the null
reproduces the production measurement exactly at zero shift and the effect-size numbers
themselves, are verifiable from the code and are not in question here.

The trimodal concentration shape looks mechanical, while the supra/infra asymmetry looks
real. The {0, 0.5, 1} mass is what the equal-sized 45%/45% bands with a 10% excluded
middle force for {infra-only, full-span, supra-only} OFFs, so any size mixture of small
single-band and full-span OFFs is trimodal with no depth structure at all; and where
`gap_chans <= 5` the final 5-channel closing can bridge the granular gap and manufacture
part of the central peak. The Wasserstein attribution sits below 1 because it correctly
counts the genuine supra/infra imbalance, which the symmetric depth null cannot make, as
a real departure. That imbalance is the laminar signal.

COM trimodality looks like a centroid-contraction artifact rather than a trimodal
occurrence pattern. COM is the per-OFF footprint midpoint, and within a bounded probe
the midpoint of a width-`w` OFF is confined to `[w/2, L-w/2]`, so large OFFs are forced
toward the centre, manufacturing a central COM peak with no corresponding peak in the
actual depth coverage. The three-marginals overlay shows this directly: COM is peakier
than either occupancy for the same OFFs. The feasible null makes this concrete: it pins
each footprint among its in-bounds positions (full-span OFFs have none), so it is the
pure centroid-contraction prediction.

Measured against that contraction baseline, the empirical COM carries a modest,
direction-consistent residual. Across structures the empirical COM is ~10-20% more
spread toward the depth extremes than the feasible null predicts (`com_spread_ratio` > 1
in every structure), with an earth-mover effect size of a few percent of the probe span:
OFFs reach extreme depths somewhat more than size and in-bounds placement alone would
produce. The Wasserstein skill-score attribution is not a valid summary for the feasible
COM, since its `W1(emp,flat)` denominator collapses when the empirical COM is itself
near-flat, as on the tall probes, sending the score wildly negative. The raw `W1` plus
the spread ratio are the honest readouts, which is why the earlier uniform-null
"attribution ~ 0" framing was dropped.

The occupancy marginals depart from the null by a real, modest amount when read against
the feasible null. H0 is OFFs of the observed sizes placed at random in-detection depth,
and its expected occupancy is the detection-edge-tapered envelope, not a flat line. The
uniform null instead places same-size OFFs over the full anatomical structure, which
extends past the detection window, and observes only the detected channels, replacing
that detection-edge taper with the asymmetric leak-in shape. Its out-of-window overhang
is divided out by unit-mass renormalization, so the f-dependent partial-visibility
deficit biases neither its TV nor its W1, both pure shape distances; it gives a
structure-dependent W1 that can be much larger where the detection window sits off-centre
in the structure, VO for instance. Against the feasible null, the in-detection no-clip
baseline, the occupancy departure is direction-consistent: an earth-mover distance of
order one to two channel pitches, concentrated in the interior. Both readouts are
significantly different (`occ_count_feasible_p` ~ 0, unavoidable at ~10^5 OFFs), so read
the effect size. `count`-occupancy, duration-blind and the most sensitive to where
events occur, localizes the laminar preference; `time`-occupancy is smeared by long and
tall OFFs. Whether the residual is biology or a detection rate that varies with depth is
the deferred follow-up, a detection-level MUA null.

The two measures are two projections of one object, the per-OFF depth marginal, so their
mutual correlation is induced and is not independent corroboration. The group-level
inference, with subject as the random effect, lives in the r-offp
`scripts/depth_profile_summary.R` companion fed by
`off-analysis export-depth-profile-summary`.
